In [ ]:
# import settings and functions
%run ./../../dataset_creation_imports.ipynb

## What mesh?

Copy your choice to the next cell

for SquareTop:
```
analytical_solution_tag = "-ana_square_top"
generate_config = generateConfig_squareTop
generate_mesh = generateMesh_squareTop
```

for SquareSinCos:
```
analytical_solution_tag = "-ana_square_sincos"
generate_config = generateConfig_squareSinCos
generate_mesh = generateMesh_squareSinCos
```

In [ ]:
# Change according to instruction above
# analytical_solution_tag = "-ana_square_sincos"
# generate_config = generateConfig_squareSinCos
# generate_mesh = generateMesh_squareSinCos

# analytical_solution_tag = "-ana_L_shape"
# generate_config = generateConfig_Lshape
# generate_mesh = generateMesh_Lshape

analytical_solution_tag = "-ana_mexi_hat"
generate_config = generateConfig_squareMexiHat
generate_mesh = generateMesh_squareMexiHat

# analytical_solution_tag = "-ana_square_top"
# generate_config = generateConfig_squareTop
# generate_mesh = generateMesh_squareTop

In [ ]:
exe = data_driven_diffusion_snes
prefix = "DD_"

exe = hdiv_data_driven_diffusion_snes
prefix = "DD_hdiv_"

# # paper:
# prefix = "DD_DD_hdiv_"

how_many_values = [11, 101, 1001, 10001]
# how_many_values = [10, 100, 1000, 10000]
# how_many_values = [11, 101, 1001]

run_simulation = True
run_line_simulation = True

run_simulation = False
run_line_simulation = False


In [ ]:
how_many = 41
#
params.conductivity = 1.0 # linear conductivity
params.element_size = 0.05 # element size in the regular mesh
params.order = 2 # approximation order for displacements
# Pre-processing parameters
params.mesh_file = "square_top"
params.length_x = 1
params.length_y = 1

params.nproc = 1 # number of processors

In [ ]:
# start display for showing results
display = Display(backend="xvfb", visible=False, size=(1024, 768))
display.start()

In [ ]:
# Testing mesh generation

params.show_mesh = True
generate_config(params)
generate_mesh(params)

In [ ]:
dataset_name = "grid_data.csv"
make_grid_linear_dataset(9, how_many, dataset_name, plot_dataset=True)

In [ ]:
# Testing running analysis
!rm out_*

use_line = "-use_line"
use_line = ""

how_many = 41
make_grid_linear_dataset(9, how_many, dataset_name, plot_dataset=False)

params.part_file = params.mesh_file + "_" + str(params.nproc) + "p.h5m"
!{mofem_part} -my_file {params.mesh_file + ".h5m"} -my_nparts {params.nproc} -output_file {params.part_file} -dim 2 -adj_dim 1
!{exe} -file_name {params.part_file} -my_order {params.order} {analytical_solution_tag} -csv_tree_file {dataset_name} -write_long_error_file {use_line} -print_integ

out_to_vtk = !ls -c1 out_result_*h5m

!convert.py {out_to_vtk[0]}

In [ ]:

params.show_file = "out_iteration_"
params.show_file = "out_result_"
params.show_field = "T"
# params.warp_factor = 0.4  # warp factor
params.show_edges = True
# params.p_save = "run_test_p.pdf"
show_results(params)

In [ ]:
params.show_field = "Q"
params.field_part = 1
show_results(params)

In [ ]:
def show_results(params):
    out_to_vtk = !ls -c1 {params.show_file}*vtk
    last_file = out_to_vtk[0]

    p = pv.Plotter(notebook=True)

    mesh = pv.read(last_file[:-3] + "vtk")
    warped_mesh = mesh

    if params.warp_field_scalar:
        warped_mesh = mesh.warp_by_scalar(scalars=params.warp_field_scalar, factor=params.warp_factor)

    if params.warp_field_vector:
        vector_magnitude = np.linalg.norm(mesh.point_data[params.warp_field_vector], axis=1)
        mesh.point_data['vector_magnitude'] = vector_magnitude
        warped_mesh = mesh.warp_by_scalar(scalars='vector_magnitude', factor=params.warp_factor)

    if params.show_edges:
        warped_mesh = warped_mesh.shrink(0.95)

    jupyter_backend = 'ipygany'

    if params.show_field in warped_mesh.point_data:
        data_location = warped_mesh.point_data
    elif params.show_field in warped_mesh.cell_data:
        data_location = warped_mesh.cell_data
    else:
        raise KeyError(f"Field '{params.show_field}' not found in point or cell data.")

    if params.field_part < 0:
        scalars = data_location[params.show_field]
    elif params.field_part == 10:  # plot gradient
        warped_mesh = warped_mesh.compute_derivative(scalars=params.show_field, preference='point')
        scalars = warped_mesh.point_data['gradient']
    else:
        scalars = warped_mesh.point_data[params.show_field][:, params.field_part]

    if params.show_field_2 and params.show_field_2 in warped_mesh.point_data:
        if params.field_part < 0:
            scalars2 = warped_mesh.point_data[params.show_field_2]
        else:
            scalars2 = warped_mesh.point_data[params.show_field_2][:, params.field_part]
        scalars = scalars + scalars2

    if params.show_field_3 and params.show_field_3 in warped_mesh.point_data:
        if params.field_part < 0:
            scalars3 = warped_mesh.point_data[params.show_field_3]
        else:
            scalars3 = warped_mesh.point_data[params.show_field_3][:, params.field_part]
        scalars = scalars + scalars3

    scalars = scalars * params.show_field_scale

    if params.show_ori_shape:
        p.add_mesh(mesh, component=None, smooth_shading=True, opacity=0.5, color='gray')

    scalar_bar_args = p_settings(p, params)

    p.add_mesh(
        warped_mesh,
        scalars=scalars,
        component=None,
        smooth_shading=False,
        cmap=params.p_cmap,
        clim=params.clim,
        show_scalar_bar=params.show_scalar_bar,
        opacity=0.9,
        scalar_bar_args=scalar_bar_args,
    )

    # Add marker at specified point if provided
    if hasattr(params, 'show_point') and params.show_point:
        point_coords = np.array(params.show_point)
        if len(point_coords) == 3:
            # add_triangle_marker(p, point_coords, size=0.05, color='magenta')
            p.add_points(
                point_coords.reshape(1, 3),
                color='black',  # Mark the point in red
                point_size=40,
                render_points_as_spheres=True,
                name='marker_point'
            )
        else:
            raise ValueError("Invalid point coordinates. Must be [x, y, z].")

    p.camera_position = params.camera_position

    p.enable_parallel_projection()
    p.enable_image_style()

    p.show(jupyter_backend=jupyter_backend)

    p_save_crop(p, params)


In [ ]:
params.show_point = [ 0.05915762 ,-0.05457881 , 0.        ]
params.show_point = [ 0.06246434, -0.07492867 , 0.        ]

In [ ]:
params.show_field = "T"
params.p_cmap = color_temperature
params.clim = (0, 1)
params.p_resolution = (500*3, 550*3)
params.font_page_part = 1./2.
params.p_save = "c5_T_marked_journey.pdf"
params.field_part = -1
show_results(params)

In [ ]:
# point_file = "out_integ_pts_6"

# !mbconvert {point_file + ".h5m"} {point_file + ".vtk"}

# params.show_file = "out_integ_pts_"
# params.show_field = "GRAD(P)_STAR"
# params.field_part = 1
# # params.warp_factor = 0.4  # warp factor
# params.show_edges = True
# # params.p_save = "run_test_p.pdf"
# show_resulting_points(params)

In [ ]:
# params.show_field = "T"
# params.field_part = -1
# # params.warp_factor = 0.4  # warp factor
# # params.p_save = "run_test_p.pdf"
# show_resulting_points(params)

In [ ]:
def save_point_fields(vtk_file, point_index, csv_file):
    # Read the .vtk file
    mesh = pv.read(vtk_file)

    # Get the point
    point = mesh.points[point_index]

    # Get the fields (scalars) of the point
    fields = mesh.point_data

    # Prepare data for saving
    data_to_save = {field: fields[field][point_index] for field in fields}

    # Save the fields to a .csv file
    with open(csv_file, 'w') as f:
        writer = csv.DictWriter(f, fieldnames=data_to_save.keys())
        writer.writeheader()

In [ ]:
import csv
import glob
import pandas as pd
import matplotlib.pyplot as plt
import pyvista as pv

!convert.py out_integ_pts_*

def get_point_fields(directory, point_index):
    # Get the list of .vtk files
    vtk_files = sorted(glob.glob(f"{directory}/out_integ_pts_*.vtk"))

    # Initialize a DataFrame to store the fields
    df = None

    for vtk_file in vtk_files:
        # Read the .vtk file
        mesh = pv.read(vtk_file)

        # Get the fields (scalars) of the point
        fields = mesh.point_data

        # Get the coordinates of the point
        point_coords = mesh.points[point_index]
        print(f"Point coordinates: {point_coords}")

        # Prepare data for saving
        data_to_save = {}
        for field in fields:
            if np.isscalar(fields[field][point_index]):
                # Scalar field
                data_to_save[field] = fields[field][point_index]
            elif len(fields[field][point_index].shape) == 1:
                # Vector field
                for i in range(fields[field][point_index].shape[0]):
                    data_to_save[f"{field}_{i}"] = fields[field][point_index][i]

        # Add the file number to the data
        file_number = int(vtk_file.split('_')[-1].split('.')[0])
        data_to_save['file_number'] = file_number

        # Append the data to the DataFrame
        if df is None:
            df = pd.DataFrame([data_to_save])
        else:
            df = pd.concat([df, pd.DataFrame([data_to_save])], ignore_index=True)

    return df

# plot_point_fields(".", 20)
df_gauss_vlaues = get_point_fields(".", 20)


In [ ]:
# df_gauss_vlaues = get_point_fields(".", 3139)
df_gauss_vlaues = get_point_fields(".", 2047)
df_gauss_vlaues = get_point_fields(".", 2775)
df_gauss_vlaues = get_point_fields(".", 2745)
df_gauss_vlaues = get_point_fields(".", 5477)
print(df_gauss_vlaues)

In [ ]:
dataset_name = "grid_data.csv"
grid_data = pd.read_csv(dataset_name)

In [ ]:
# Interleave the points from 'fields' and 'stars'
x_values = np.ravel(np.column_stack((df_gauss_vlaues['GRAD(P)_0'], df_gauss_vlaues['GRAD(P)_STAR_0'])))
y_values = np.ravel(np.column_stack((df_gauss_vlaues['Q_0'], df_gauss_vlaues['Q_STAR_0'])))


# Calculate the differences between consecutive points
dx = np.diff(x_values)
dy = np.diff(y_values)

# Calculate the length of each line segment
lengths = np.hypot(dx, dy)

plt.figure(figsize=[5,4])
plt.scatter(grid_data['gradx'], grid_data['fluxx'], label='Material dataset', zorder=1, s=2)
plt.scatter(df_gauss_vlaues['GRAD(P)_0'], df_gauss_vlaues['Q_0'], label='Field values', marker='x', zorder=1)
plt.scatter(df_gauss_vlaues['GRAD(P)_STAR_0'], df_gauss_vlaues['Q_STAR_0'], label='Closest data point', marker='*', zorder=1, color='b')

# Normalize the lengths to get widths
widths = lengths / np.max(lengths) * 0.005  # adjust the factor as needed

# Plot each arrow individually with its own width
for i in range(len(x_values[:-1])):
    plt.quiver(x_values[i], y_values[i], dx[i], dy[i], scale_units='xy', angles='xy', scale=1, color='black', width=widths[i], zorder=2)

# plt.xlabel('Greadient')
# plt.ylabel('Flux')
plt.xlabel(label_gradient_g_x)
plt.ylabel(label_flux_x)
# plt.xlim(-0.5,8)
# plt.ylim(-8,0.5)
ax = plt.gca()  # Define ax variable
ax.grid(True, ls=':')  # Call ax.grid() method
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('integration_journey_x.pdf')
plt.show()


In [ ]:
# Interleave the points from 'fields' and 'stars'
x_values = np.ravel(np.column_stack((df_gauss_vlaues['GRAD(P)_1'], df_gauss_vlaues['GRAD(P)_STAR_1'])))
y_values = np.ravel(np.column_stack((df_gauss_vlaues['Q_1'], df_gauss_vlaues['Q_STAR_1'])))


# Calculate the differences between consecutive points
dx = np.diff(x_values)
dy = np.diff(y_values)

# Calculate the length of each line segment
lengths = np.hypot(dx, dy)

plt.figure(figsize=[5,4])
plt.scatter(grid_data['grady'], grid_data['fluxy'], label='Material dataset', zorder=1, s=2)
plt.scatter(df_gauss_vlaues['GRAD(P)_1'], df_gauss_vlaues['Q_1'], label='Field values', marker='x', zorder=1)
plt.scatter(df_gauss_vlaues['GRAD(P)_STAR_1'], df_gauss_vlaues['Q_STAR_1'], label='Closest data point', marker='*', zorder=1, color='b')

# Normalize the lengths to get widths
widths = lengths / np.max(lengths) * 0.005  # adjust the factor as needed

# Plot each arrow individually with its own width
for i in range(len(x_values[:-1])):
    plt.quiver(x_values[i], y_values[i], dx[i], dy[i], scale_units='xy', angles='xy', scale=1, color='black', width=widths[i], zorder=2)

# plt.xlabel('Greadient')
# plt.ylabel('Flux')
plt.xlabel(label_gradient_g_y)
plt.ylabel(label_flux_y)
# plt.xlim(-0.5,8)
# plt.ylim(-8,0.5)
ax = plt.gca()  # Define ax variable
ax.grid(True, ls=':')  # Call ax.grid() method
plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig('integration_journey_y.pdf')
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import matplotlib.pyplot as plt

# Interleave the points from 'fields' and 'stars' for X and Y components
x_values = np.ravel(np.column_stack((df_gauss_vlaues['GRAD(P)_0'], df_gauss_vlaues['GRAD(P)_STAR_0'])))
y_values = np.ravel(np.column_stack((df_gauss_vlaues['GRAD(P)_1'], df_gauss_vlaues['GRAD(P)_STAR_1'])))
z_values = np.ravel(np.column_stack((df_gauss_vlaues['Q_0'], df_gauss_vlaues['Q_STAR_0'])))

# Calculate differences for quivers
dx = np.diff(x_values)
dy = np.diff(y_values)
dz = np.diff(z_values)

# Calculate the length of each line segment
lengths = np.hypot(np.hypot(dx, dy), dz)

# Normalise the lengths to get widths for quivers
widths = lengths / np.max(lengths) * 0.01  # Adjust the factor if needed

# Create 3D figure
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Scatter plots for dataset, field values, and closest points
ax.scatter(grid_data['gradx'], grid_data['grady'], grid_data['fluxx'], label='Material dataset', s=2, zorder=1)
ax.scatter(df_gauss_vlaues['GRAD(P)_0'], df_gauss_vlaues['GRAD(P)_1'], df_gauss_vlaues['Q_0'], label='Field values', marker='x', zorder=1)
ax.scatter(df_gauss_vlaues['GRAD(P)_STAR_0'], df_gauss_vlaues['GRAD(P)_STAR_1'], df_gauss_vlaues['Q_STAR_0'], label='Closest data point', marker='*', color='b', zorder=1)

# Plot quivers in 3D with varying widths
for i in range(len(x_values[:-1])):
    ax.quiver(x_values[i], y_values[i], z_values[i], 
              dx[i], dy[i], dz[i], 
              color='black', length=1, arrow_length_ratio=0.2, zorder=2)

# Labels and grid
ax.set_xlabel(r'$g_x^*$', labelpad=10)
ax.set_ylabel(r'$g_y^*$', labelpad=10)
ax.set_zlabel(r'$q_x^*$', labelpad=10)
ax.grid(True, linestyle=':')

# Legend and layout
# leg = ax.legend()
# leg.set_zorder(10) 
ax.legend(loc='upper right', bbox_to_anchor=(0.2, 0.95), fontsize=12)
plt.tight_layout()
plt.savefig('integration_journey_3D.pdf')
plt.show()


In [ ]:
data = pd.read_csv('errors_long_file.csv', header=0,  index_col=False)

In [ ]:
# fig, ax = plt.subplots()
# ax.plot(data['rmsErr_long_file'], label=r'Temperature error')
# ax.plot(data[' rmsGradErr_long_file'], label=r'Gradient error')
# ax.plot(data[' rmsFluxErr_long_file'], label=r'Flux error')
# ax.plot(data[' rmsPointDistErr_long_file'], label=r'Point distance', ls='--', color='black', marker='o')
# ax.set_yscale('log')
# ax.grid(True, ls=':')
# ax.legend()

## Plot with line and with dataset

In [ ]:
main_error_file = "errors_long_file.csv"

line_errors = prefix+"line_errors.csv"
data_errors = "data_errors.csv"

use_line = "-use_line"
use_line = ""

how_many = 101
make_grid_linear_dataset(9, how_many, dataset_name, plot_dataset=False)

params.part_file = params.mesh_file + "_" + str(params.nproc) + "p.h5m"
!{mofem_part} -my_file {params.mesh_file + ".h5m"} -my_nparts {params.nproc} -output_file {params.part_file} -dim 2 -adj_dim 1
# # !{classic_diffusion} -file_name {params.part_file} -my_order {params.order} {analytical_solution_tag}
# !{data_driven_diffusion_snes} -file_name {params.part_file} -my_order {params.order} {analytical_solution_tag} -csv_tree_file {dataset_name} -write_long_error_file {use_line}

# !mv {main_error_file} {data_errors}


# Create an empty DataFrame to store the data
all_data = []

# Define the range of values for how_many
# how_many_values = [11, 101, 1001, 10001]

if run_simulation:

    for how_many in how_many_values:
        !rm {main_error_file}

        data_errors = prefix+'data_errors'+str(how_many)+'.csv'
        print(data_errors)

        make_grid_linear_dataset(9, how_many, dataset_name, plot_dataset=False)

        !{exe} -file_name {params.part_file} -my_order {params.order} {analytical_solution_tag} -csv_tree_file {dataset_name} -write_long_error_file {use_line}
        
        # Read the data from the main_error_file
        data = pd.read_csv(main_error_file)
        
        # Append the data to the all_data DataFrame
        all_data.append(data)
        !mv {main_error_file} {data_errors}
        print("data errors file: ", data_errors)


In [ ]:
if run_simulation == False:
    for how_many in how_many_values:
            data_errors = prefix+'data_errors'+str(how_many)+'.csv'
            data = pd.read_csv(data_errors)
            all_data.append(data)

In [ ]:
if run_line_simulation:
    use_line = "-use_line"
    !{exe} -file_name {params.part_file} -my_order {params.order} {analytical_solution_tag} -csv_tree_file {dataset_name} -write_long_error_file {use_line}
    print(line_errors)
    !mv {main_error_file} {line_errors}

In [ ]:
# if run_line_simulation == False:
#     data = pd.read_csv(line_errors)
#     all_data.append(data)

In [ ]:
# !mv {main_error_file} {line_errors}
line_pd = pd.read_csv(line_errors, header=0,  index_col=False)

error = 'rmsErr_long_file'
label = r'Temperature error'

def plot_error(error, label):

    fig, ax = plt.subplots()

    for i in range(len(all_data)):
        ax.plot(all_data[i][error], label=f"{how_many_values[i]}$^4$ points")

    ax.plot(line_pd[error], label=r'line equation', ls=':', color='black')
    ax.set_ylabel(label)
    ax.set_xlabel(r'Iteration')
    ax.set_yscale('log')
    ax.grid(True, ls=':')
    ax.legend()
    plt.tight_layout()
    plt.savefig(prefix+'iteration_'+error+'.pdf')

plot_error('rmsErr_long_file', r'Temperature error')
plot_error(' rmsGradErr_long_file', r'Gradient error')
plot_error(' rmsFluxErr_long_file', r'Flux error')
plot_error(' rmsPointDistErr_long_file', r'Point distance')

In [ ]:
# line_pd = pd.read_csv(line_errors, header=0,  index_col=False)
# data_pd = pd.read_csv(data_errors, header=0,  index_col=False)

# fig, ax = plt.subplots()
# ax.plot(data_pd['rmsErr_long_file'], label=r'Temperature error')
# ax.plot(data_pd[' rmsGradErr_long_file'], label=r'Gradient error')
# ax.plot(data_pd[' rmsFluxErr_long_file'], label=r'Flux error')
# ax.plot(data_pd[' rmsPointDistErr_long_file'], label=r'Point distance', ls='--', color='black', marker='o')
# ax.set_ylabel(r'Global RMS error')
# ax.set_xlabel(r'Iteration')
# ax.set_yscale('log')
# ax.grid(True, ls=':')
# y_range = ax.get_ylim()
# ax.legend()
# plt.tight_layout()
# fig.savefig("DD_small_data_errors_with_iterations.pdf")

# fig, ax = plt.subplots()
# ax.plot(line_pd['rmsErr_long_file'], label=r'Temperature error')
# ax.plot(line_pd[' rmsGradErr_long_file'], label=r'Gradient error')
# ax.plot(line_pd[' rmsFluxErr_long_file'], label=r'Flux error')
# ax.plot(line_pd[' rmsPointDistErr_long_file'], label=r'Point distance', ls='--', color='black', marker='o')
# ax.set_ylabel(r'Global RMS error')
# ax.set_xlabel(r'Iteration')
# ax.set_yscale('log')
# ax.grid(True, ls=':')
# ax.legend()
# ax.set_ylim(y_range)
# plt.tight_layout()
# fig.savefig("DD_line_errors_with_iterations.pdf")


In [ ]:
DD_data = []
line_data = []

all_data = []
prefix = "DD_"
for how_many in how_many_values:
        data_errors = prefix+'data_errors'+str(how_many)+'.csv'
        data = pd.read_csv(data_errors)
        all_data.append(data)
line_pd = pd.read_csv(prefix+"line_errors.csv", header=0,  index_col=False)

DD_data.append(all_data)
line_data.append(line_pd)

all_data = []
prefix = "DD_hdiv_"
for how_many in how_many_values:
        data_errors = prefix+'data_errors'+str(how_many)+'.csv'
        data = pd.read_csv(data_errors)
        all_data.append(data)
line_pd = pd.read_csv(prefix+"line_errors.csv", header=0,  index_col=False)

DD_data.append(all_data)
line_data.append(line_pd)

In [ ]:
print(line_data)


In [ ]:

prefix = "DD_DD_hdiv_"

def plot_error(error, label):

    fig, ax = plt.subplots()

    labels_2 = ['Stronger DD', 'Weaker DD']
    line_types = ['-', '--']
    markers = ['o', 'x']

    for j in range(len(DD_data)):
        all_data = DD_data[j]

        # reset colours
        ax.set_prop_cycle(None)

        for i in range(len(all_data)):
            ax.plot(all_data[i][error], label=labels_2[j]+f"; {how_many_values[i]}$^4$ points", ls=line_types[j], marker=markers[j],markerfacecolor='none')
        ax.plot(line_data[j][error], label=labels_2[j]+r'; line equation', ls=":", color='black', marker=markers[j],markerfacecolor='none')

    # for i in range(len(all_data)):
    #     ax.plot(all_data[i][error], label=f"{how_many_values[i]}$^4$ points")

    # ax.plot(line_pd[error], label=r'line equation', ls=':', color='black')
    ax.set_ylabel(label)
    ax.set_xlabel(r'Iteration')
    ax.set_yscale('log')
    ax.grid(True, ls=':')
    # decrease fond size of legend
    ax.legend(prop={'size': 8})
    plt.tight_layout()
    plt.savefig(prefix+'iteration_'+error+'.pdf')
    # save svg
    plt.savefig(prefix+'iteration_'+error+'.svg')

error_label_list = [(r'Global error temperature $T$ $L^2$-norm'),
               (r'Global error gradient $\mathbf g$ $L^2$-norm'), (r'Global error flux $\mathbf q$ $L^2$-norm')]

plot_error('rmsErr_long_file', r'Global error $T$ $L^2$-norm')
plot_error(' rmsGradErr_long_file', r'Gradient error')
plot_error(' rmsFluxErr_long_file', r'Global error $\mathbf q$ $L^2$-norm')
plot_error(' rmsPointDistErr_long_file', r'Point distance $\varepsilon_d\left(T, \mathbf g, \mathbf q\right)$')